# 2. Feature Engineering & Offline Augmentation

Aligned with ETASR 2026 (Zennou et al.).
- Dataset is filtered to exactly **1440 Speech files**.
- A **Random 80-10-10 Split** is applied to simulate the paper's data setup.
- **Data Augmentation** (Noise, Time Stretch, Pitch Shift) is strictly applied *offline* to the training set only.

In [1]:
import os, glob
import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import IPython.display as ipd
from sklearn.preprocessing import StandardScaler
import joblib
from tqdm.notebook import tqdm
import random

# Seed for reproducibility
np.random.seed(42)
random.seed(42)

# Constants
TARGET_SR    = 22050
DURATION     = 3.0                     
N_SAMPLES    = int(TARGET_SR * DURATION)
FRAME_LENGTH = int(0.025 * TARGET_SR)  
HOP_LENGTH   = int(0.010 * TARGET_SR)  
N_FFT        = 2048                    
N_MFCC       = 13                      
MAX_FRAMES   = 400                     
N_FEATURES   = N_MFCC * 3 + 1         


## 1. Filter Dataset (1440 Speech Files) & Random Split

In [2]:
DATA_RAW = '../data/raw'
audio_files = glob.glob(os.path.join(DATA_RAW, '**', '*.wav'), recursive=True)

emotion_mapping = {
    1: 'neutral', 2: 'calm',    3: 'happy',    4: 'sad',
    5: 'angry',   6: 'fearful', 7: 'disgust',  8: 'surprised'
}

data = []
for file_path in audio_files:
    parts = os.path.basename(file_path).replace('.wav', '').split('-')
    if len(parts) == 7:
        modality, vocal_channel, emotion, intensity, statement, repetition, actor = parts
        # Filter ONLY Speech (modality == 03)
        if modality == '03':
            data.append({
                'file_path':     file_path,
                'emotion_label': emotion_mapping.get(int(emotion), 'unknown'),
            })

df = pd.DataFrame(data)
print(f"Total Speech files found: {len(df)} (Expected: 1440)")

# Random Shuffle & Split (80% Train, 10% Val, 10% Test)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
n_total = len(df)
n_train = int(0.8 * n_total)
n_val   = int(0.1 * n_total)

df['split'] = ['train'] * n_train + ['val'] * n_val + ['test'] * (n_total - n_train - n_val)
os.makedirs('../data/processed', exist_ok=True)
df.to_csv('../data/processed/manifest_random.csv', index=False)
display(df.groupby('split').size())

Total Speech files found: 2880 (Expected: 1440)


split
test      288
train    2304
val       288
dtype: int64

## 2. Feature Extraction & Augmentation Functions

In [3]:
def preprocess_audio(file_path: str) -> np.ndarray:
    """Load, resample, pre-emphasise, trim silence, and fix length."""
    y, sr = librosa.load(file_path, sr=None, mono=True)

    # 1. Resample to TARGET_SR
    if sr != TARGET_SR:
        y = librosa.resample(y, orig_sr=sr, target_sr=TARGET_SR)

    # 2. Pre-emphasis filter
    y = librosa.effects.preemphasis(y, coef=0.97)

    # 3. Trim leading/trailing silence
    y, _ = librosa.effects.trim(y, top_db=30)

    # 4. Pad or truncate to fixed duration
    if len(y) > N_SAMPLES:
        y = y[:N_SAMPLES]
    else:
        y = np.pad(y, (0, N_SAMPLES - len(y)), mode="constant")

    return y


def extract_features(y: np.ndarray, sr: int = TARGET_SR) -> np.ndarray:
    """
    Extract MFCC + Delta + Delta-Delta + RMSE features.

    Returns
    -------
    np.ndarray of shape (MAX_FRAMES, N_FEATURES)
        N_FEATURES = 13+13+13+1 = 40
    """
    # 1. MFCC (13 coefficients)
    mfcc = librosa.feature.mfcc(
        y=y, sr=sr, n_mfcc=N_MFCC,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        win_length=FRAME_LENGTH,
    )  # shape: (13, T)

    # 2. Delta MFCC
    delta = librosa.feature.delta(mfcc, width=9)       # (13, T)

    # 3. Delta-Delta MFCC
    delta2 = librosa.feature.delta(mfcc, order=2, width=9)  # (13, T)

    # 4. RMSE (root mean square energy) per frame
    rmse = librosa.feature.rms(
        y=y, frame_length=FRAME_LENGTH, hop_length=HOP_LENGTH
    )  # shape: (1, T_rmse)  — may differ by ±1 frame from MFCC

    # Align RMSE length to MFCC frame count
    T_mfcc = mfcc.shape[1]
    if rmse.shape[1] > T_mfcc:
        rmse = rmse[:, :T_mfcc]
    elif rmse.shape[1] < T_mfcc:
        rmse = np.pad(rmse, ((0, 0), (0, T_mfcc - rmse.shape[1])), mode="edge")

    # Concatenate → (40, T)
    features = np.concatenate([mfcc, delta, delta2, rmse], axis=0)

    # Transpose → (T, 40)
    features = features.T

    # Zero-pad or truncate to MAX_FRAMES (400)
    T = features.shape[0]
    if T >= MAX_FRAMES:
        features = features[:MAX_FRAMES, :]
    else:
        pad = np.zeros((MAX_FRAMES - T, N_FEATURES), dtype=np.float32)
        features = np.vstack([features, pad])

    return features.astype(np.float32)   # (400, 40)


In [4]:
def add_noise(y: np.ndarray, snr_db: float = 20.0) -> np.ndarray:
    """Add Gaussian noise at a given SNR (dB)."""
    signal_power = np.mean(y ** 2)
    noise_power  = signal_power / (10 ** (snr_db / 10))
    noise = np.random.normal(0, np.sqrt(noise_power), size=y.shape)
    return (y + noise).astype(np.float32)


def time_stretch(y: np.ndarray,
                 rate_min: float = 0.8,
                 rate_max: float = 1.2) -> np.ndarray:
    """Randomly stretch / compress the time axis without affecting pitch."""
    rate = np.random.uniform(rate_min, rate_max)
    return librosa.effects.time_stretch(y.astype(np.float32), rate=rate)


def time_shift(y: np.ndarray, sr: int = 22050,
               shift_max: float = 0.5) -> np.ndarray:
    """Randomly shift the signal left or right by up to shift_max seconds."""
    shift = int(np.random.uniform(-shift_max, shift_max) * sr)
    return np.roll(y, shift).astype(np.float32)


def pitch_shift(y: np.ndarray, sr: int = 22050,
                semitones_max: float = 2.0) -> np.ndarray:
    """Randomly shift pitch by ±semitones_max semitones."""
    n_steps = np.random.uniform(-semitones_max, semitones_max)
    return librosa.effects.pitch_shift(
        y.astype(np.float32), sr=sr, n_steps=n_steps
    )


# ── Augmentation strategy ─────────────────────────────────────────────────────

# Each entry: (function, kwargs, probability)
AUGMENTATIONS = [
    (add_noise,    {"snr_db": 20.0},                          0.5),
    (time_stretch, {"rate_min": 0.85, "rate_max": 1.15},      0.4),
    (time_shift,   {"sr": 22050, "shift_max": 0.3},           0.4),
    (pitch_shift,  {"sr": 22050, "semitones_max": 2.0},       0.3),
]

def augment(y: np.ndarray, sr: int = 22050,
            p_apply: float = 1.0) -> np.ndarray:
    """
    Apply a random subset of augmentations to an audio array.

    Parameters
    ----------
    y        : Raw audio signal (float32 array)
    sr       : Sample rate (default 22050)
    p_apply  : Overall probability of applying any augmentation at all.
               Set to 1.0 to always augment (typical for training).

    Returns
    -------
    Augmented audio signal (same dtype as input, length may vary slightly).
    """
    if np.random.random() > p_apply:
        return y

    y_aug = y.copy()
    for fn, kwargs, prob in AUGMENTATIONS:
        if np.random.random() < prob:
            try:
                y_aug = fn(y_aug, **kwargs)
            except Exception:
                # Fallback: skip this augmentation if it fails
                pass

    return y_aug.astype(np.float32)


## 3. Run Offline Extraction

For the **train** split, we generate 4 augmented versions for every original file. This drastically expands the dataset to prevent overfitting.

In [5]:
FEAT_DIR = "../data/processed/features_random"
AUGMENT_COPIES = 4

emotion_classes = sorted(df['emotion_label'].unique())
label2idx = {e: i for i, e in enumerate(emotion_classes)}

for split in ['train', 'val', 'test']:
    split_df = df[df['split'] == split].reset_index(drop=True)
    out_dir  = os.path.join(FEAT_DIR, split)
    os.makedirs(out_dir, exist_ok=True)
    scaler_path = os.path.join(FEAT_DIR, 'train', 'scaler.pkl')

    X_list, y_list = [], []
    print(f"\n── Extracting {split} split ({len(split_df)} base files) ──")

    for _, row in tqdm(split_df.iterrows(), total=len(split_df)):
        y_audio = preprocess_audio(row['file_path'])
        label_idx = label2idx[row['emotion_label']]
        
        # 1. Clean feature
        X_list.append(extract_features(y_audio))
        y_list.append(label_idx)

        # 2. Augmented features (only for training)
        if split == 'train':
            for _ in range(AUGMENT_COPIES):
                y_aug = augment(y_audio)
                X_list.append(extract_features(y_aug))
                y_list.append(label_idx)

    X = np.array(X_list, dtype=np.float32)
    y = np.array(y_list,  dtype=np.int32)
    print(f"  Raw shape: X={X.shape}, y={y.shape}")

    N, T, F = X.shape
    X_flat = X.reshape(-1, F)

    if split == 'train':
        scaler = StandardScaler().fit(X_flat)
        joblib.dump(scaler, scaler_path)
    else:
        scaler = joblib.load(scaler_path)

    X_norm = scaler.transform(X_flat).reshape(N, T, F).astype(np.float32)
    np.save(os.path.join(out_dir, 'X.npy'), X_norm)
    np.save(os.path.join(out_dir, 'y.npy'), y)
    joblib.dump({'label2idx': label2idx}, os.path.join(out_dir, 'label_encoder.pkl'))
    
    print(f"  ✅ Saved normalized features -> {out_dir}")



── Extracting train split (2304 base files) ──


  0%|          | 0/2304 [00:00<?, ?it/s]

  Raw shape: X=(11520, 400, 40), y=(11520,)
  ✅ Saved normalized features -> ../data/processed/features_random/train

── Extracting val split (288 base files) ──


  0%|          | 0/288 [00:00<?, ?it/s]

  Raw shape: X=(288, 400, 40), y=(288,)
  ✅ Saved normalized features -> ../data/processed/features_random/val

── Extracting test split (288 base files) ──


  0%|          | 0/288 [00:00<?, ?it/s]

  Raw shape: X=(288, 400, 40), y=(288,)
  ✅ Saved normalized features -> ../data/processed/features_random/test
